# Face inversion: scrambled faces, both seeds

Runs the scrambled-face test (2x2, 4x4, 8x8 tiles) on all 6 checkpoints with the same test pairs and shuffles as notebook 01, and averages the two seeds. Needs the checkpoints in `MyDrive/face_inversion/ckpt/` and `split.json` in `MyDrive/face_inversion/`. Takes about 5 min on a T4.

In [1]:
!pip -q install git+https://github.com/AMLab-Amsterdam/lie_learn
!pip -q install escnn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.isdir('/content/vggface2_aligned/test'):
    !tar -xf /content/drive/MyDrive/face_inversion/vggface2_aligned112.tar -C /content vggface2_aligned/test
print('test faces ready')

Mounted at /content/drive
test faces ready


In [3]:
# scrambled-face test (2x2, 4x4, 8x8) for all 6 checkpoints, same pairs/seeds as notebook 01
import os,sys,glob,json,numpy as np,torch,cv2,torch.nn.functional as F,torchvision
from sklearn.metrics import roc_auc_score
from escnn import gspaces, nn as enn
D='cuda'
ROOTD='/content/drive/MyDrive/face_inversion'; CK=f'{ROOTD}/ckpt'; TROOT='/content/vggface2_aligned/test'
ids=json.load(open(f'{ROOTD}/split.json'))['ids']
MEAN=torch.tensor([0.485,0.456,0.406],device=D).view(1,3,1,1)*255; STD=torch.tensor([0.229,0.224,0.225],device=D).view(1,3,1,1)*255
yy,xx=torch.meshgrid(torch.arange(112,device=D),torch.arange(112,device=D),indexing='ij')
MASK=(((yy-55.5)**2+(xx-55.5)**2)<=56**2).float().view(1,1,112,112)
prep=lambda x:((x.float()-MEAN)/STD*MASK).contiguous(memory_format=torch.channels_last)
G=gspaces.rot2dOnR2(N=4); rtype=lambda n:enn.FieldType(G,n*[G.regular_repr])
MaxPool=getattr(enn,'PointwiseMaxPool2D',None) or enn.PointwiseMaxPool; EQ=97
class EqBlock(torch.nn.Module):
    def __init__(s,tin,tout,stride):
        super().__init__()
        s.c1,s.b1,s.r1=enn.R2Conv(tin,tout,3,stride=stride,padding=1,bias=False),enn.InnerBatchNorm(tout),enn.ReLU(tout,inplace=True)
        s.c2,s.b2=enn.R2Conv(tout,tout,3,padding=1,bias=False),enn.InnerBatchNorm(tout)
        s.sc=None if (stride==1 and tin==tout) else enn.SequentialModule(enn.R2Conv(tin,tout,1,stride=stride,bias=False),enn.InnerBatchNorm(tout))
        s.r2=enn.ReLU(tout,inplace=True)
    def forward(s,x):
        y=s.b2(s.c2(s.r1(s.b1(s.c1(x))))); return s.r2(y+(x if s.sc is None else s.sc(x)))
class C4ResNet18(torch.nn.Module):
    def __init__(s,n,widths=(32,64,128,256)):
        super().__init__(); s.tin=enn.FieldType(G,3*[G.trivial_repr]); t=rtype(widths[0])
        s.stem=enn.SequentialModule(enn.R2Conv(s.tin,t,7,stride=2,padding=3,bias=False),enn.InnerBatchNorm(t),enn.ReLU(t,inplace=True),MaxPool(t,kernel_size=3,stride=2,padding=1))
        b,tp=[],t
        for i,w in enumerate(widths):
            for j in range(2): b.append(EqBlock(tp,rtype(w),2 if (i>0 and j==0) else 1)); tp=rtype(w)
        s.blocks=torch.nn.Sequential(*b); s.gpool=enn.GroupPooling(tp); s.fc=torch.nn.Linear(widths[-1],n)
def feats(m,x):
    if isinstance(m,C4ResNet18):
        x=F.interpolate(x.float(),size=EQ,mode='bilinear',align_corners=False)
        return m.gpool(m.blocks(m.stem(enn.GeometricTensor(x,m.tin)))).tensor.mean((2,3))
    x=m.maxpool(m.relu(m.bn1(m.conv1(x)))); return torch.flatten(m.avgpool(m.layer4(m.layer3(m.layer2(m.layer1(x))))),1)
@torch.no_grad()
def embed(m,imgs,bs=500):
    out=[]
    for i in range(0,len(imgs),bs):
        with torch.autocast(D,dtype=torch.float16): out.append(feats(m,prep(imgs[i:i+bs])).float())
    return F.normalize(torch.cat(out),dim=1)
def scramble(x,g,gdev):
    N,C,H,W=x.shape; t=H//g
    tiles=x.reshape(N,C,g,t,g,t).permute(0,2,4,1,3,5).reshape(N,g*g,C,t,t)
    gen=torch.Generator(device=gdev).manual_seed(0)
    perm=torch.argsort(torch.rand(N,g*g,device=gdev,generator=gen),1).to(D)
    tiles=tiles[torch.arange(N,device=D)[:,None],perm]
    return tiles.reshape(N,g,g,C,t,t).permute(0,3,1,4,2,5).reshape(N,C,H,W)
tids=sorted(d for d in os.listdir(TROOT) if d.startswith('n'))
rng=np.random.default_rng(1); items=[]
for lab,i in enumerate(tids):
    fs=sorted(glob.glob(f'{TROOT}/{i}/*.jpg')); items+=[(f,lab) for f in rng.choice(fs,min(10,len(fs)),replace=False)]
Xte=torch.stack([torch.from_numpy(cv2.cvtColor(cv2.imread(f),cv2.COLOR_BGR2RGB)).permute(2,0,1) for f,_ in items]).to(D)
Yte=[l for _,l in items]; by_id={}
for k,l in enumerate(Yte): by_id.setdefault(l,[]).append(k)
labs=list(by_id); NP=3000
SP=torch.tensor([tuple(rng.choice(by_id[labs[rng.integers(len(labs))]],2,replace=False)) for _ in range(NP)],device=D)
DP=torch.tensor([(rng.choice(by_id[labs[a]]),rng.choice(by_id[labs[b]])) for a,b in (rng.choice(len(labs),2,replace=False) for _ in range(NP))],device=D)
P=torch.cat([SP,DP]); y=np.r_[np.ones(NP),np.zeros(NP)]
def auc(m,tb=lambda z:z): return roc_auc_score(y,(embed(m,Xte[P[:,0]].float())*embed(m,tb(Xte[P[:,1]].float()))).sum(1).cpu().numpy())
print('test faces',tuple(Xte.shape),flush=True); R=[]
for mode in ['baseline','augmented','equivariant']:
    for sd in (0,1):
        m=C4ResNet18(len(ids)) if mode=='equivariant' else torchvision.models.resnet18(num_classes=len(ids))
        m.load_state_dict(torch.load(f'{CK}/{mode}{"_s1" if sd else ""}.pt',map_location='cpu',weights_only=False)['model'])
        m=m.to(D).to(memory_format=torch.channels_last).eval()
        up=auc(m); r=[mode,sd,round(up,4)]
        for g in (2,4,8):
            s=auc(m,lambda z,g=g:scramble(z,g,'cuda')); r.append(round((s-.5)/(up-.5),3))
        R.append(r); print(*r,flush=True); del m; torch.cuda.empty_cache()
import pandas as pd
df=pd.DataFrame(R,columns=['model','seed','auc_up','scr2_kept','scr4_kept','scr8_kept'])
print(df.to_string(index=False)); print('\nMEAN OF 2 SEEDS'); print(df.groupby('model',sort=False)[['scr2_kept','scr4_kept','scr8_kept']].mean().round(3))
df.to_csv(f'{ROOTD}/analysis/scramble_2seed.csv',index=False)


test faces (5000, 3, 112, 112)
baseline 0 0.8499 0.502 0.223 0.077
baseline 1 0.8396 0.515 0.229 0.09
augmented 0 0.8951 0.58 0.268 0.108
augmented 1 0.8943 0.595 0.273 0.103
equivariant 0 0.9025 0.563 0.186 0.089
equivariant 1 0.8971 0.529 0.189 0.065
      model  seed  auc_up  scr2_kept  scr4_kept  scr8_kept
   baseline     0  0.8499      0.502      0.223      0.077
   baseline     1  0.8396      0.515      0.229      0.090
  augmented     0  0.8951      0.580      0.268      0.108
  augmented     1  0.8943      0.595      0.273      0.103
equivariant     0  0.9025      0.563      0.186      0.089
equivariant     1  0.8971      0.529      0.189      0.065

MEAN OF 2 SEEDS
             scr2_kept  scr4_kept  scr8_kept
model                                       
baseline         0.508      0.226      0.083
augmented        0.587      0.270      0.106
equivariant      0.546      0.188      0.077
